# 1 Codificación de caracteres y expresiones regulares
La codificación de caracteres y las expresiones regulares son fundamentales en la minería de texto y la búsqueda de información. La correcta codificación asegura que los caracteres se interpreten y visualicen adecuadamente, mientras que las expresiones regulares permiten buscar, filtrar y manipular texto basado en patrones específicos. Si no se controla la correcta codificación de caracteres, pueden surgir problemas como la aparición de caracteres "extraños" o signos de interrogación, lo que puede llevar a la pérdida de información importante y errores en el procesamiento de datos. Además, la falta de una codificación adecuada puede causar incompatibilidades entre diferentes sistemas y aplicaciones. Juntas, estas herramientas facilitan la extracción eficiente y precisa de información relevante de grandes volúmenes de datos textuales, mejorando la calidad y la utilidad del análisis de datos.


# 2 Codificación de caracteres
## 2.1 Introducción
La codificación de caracteres es crucial en la minería de texto, especialmente cuando se manejan fuentes internacionales o alfabetos diversos (latino, cirílico, chino, japonés, árabe, etc.). Una codificación incorrecta puede resultar en caracteres "extraños" o signos de interrogación. Esta sección cubre los fundamentos de la codificación, estándares como ASCII, Unicode y UTF-8, y sus implicaciones en la programación en Python. Se proporcionarán ejemplos de lectura y escritura de archivos con diferentes codificaciones, destacando por qué UTF-8 es el estándar en la mayoría de los proyectos modernos.

## 2.2 ASCII, Unicode y UTF-8
A lo largo del tiempo, se han desarrollado varios formatos de codificación, incluyendo ASCII, Unicode y UTF-8.
- **ASCII**: Desarrollado en la década de 1960 para caracteres en inglés, utiliza 7 bits e incluye 128 códigos para letras mayúsculas/minúsculas, dígitos, signos de puntuación y caracteres de control. Por ejemplo, la letra "A" corresponde al 65, representado en binario como 01000001. Sin embargo, es insuficiente para caracteres con tildes u otros símbolos.
- **Unicode**: Busca unificar la representación de todos los alfabetos posibles, asignando un "código de punto" único a cada carácter en casi todos los idiomas y sistemas de escritura, incluyendo caracteres latinos con tildes, ideogramas chinos y emojis.
- **UTF-8**: Un esquema de codificación ampliamente adoptado que utiliza una cantidad variable de bytes (1 a 4) para representar cada carácter Unicode. Es retrocompatible con ASCII (los primeros 128 símbolos coinciden exactamente). Python utiliza UTF-8 por defecto en versiones recientes, evitando problemas de visualización como "Ã±" en lugar de "ñ".

## 2.3 Manejo de Codificaciones en Python
Python permite especificar la codificación de un archivo al abrirlo para lectura o escritura. Por ejemplo:

In [20]:
# Lectura de un archivo con codificación UTF-8
with open("test_utf8.txt", "r", encoding="utf-8") as f:
    contenido = f.read()
    print(contenido)

Hola, ¿cómo estás?
This is an additional line.
Voici une autre ligne de texte.
Hier ist eine weitere Textzeile.
这里有另一行文字。
ここに別の行があります。
여기에 또 다른 줄이 있습니다.
Это еще одна строка текста.
Aqui está outra linha de texto.



In [21]:
# Escritura en un archivo con codificación ISO-8859-1
with open("datos_latin1.csv", "w", encoding="latin-1") as f:
    f.write('''title, character
char_1, á
char_2, é
char_3, í
char_4, ó
char_5, ú
char_6, ü''')

In [22]:
# Lectura con codificación incorrecta que dará error
try:
    with open("datos_latin1.csv", "r", encoding="utf-8") as f:
        contenido = f.read()
        print(contenido)
except UnicodeDecodeError as e:
    print("Error de decodificación:", e)

Error de decodificación: 'utf-8' codec can't decode byte 0xe1 in position 25: invalid continuation byte


Si no se especifica el parámetro `encoding`, Python intentará utilizar la codificación predeterminada del sistema, lo que puede variar según la configuración regional del SO (Windows, Linux, macOS). Esto puede causar problemas al compartir código con personas en diferentes países o servidores.

Para la minería de texto, es útil cargar los datos en un DataFrame para inspección y transformación. Tanto Pandas como Polars permiten especificar la codificación:

In [23]:
import pandas as pd

df_pandas = pd.read_csv(
    "datos_latin1.csv",
    encoding="latin-1",
    sep=",",
    header=0  # La primera fila contiene los nombres de las columnas
)
df_pandas

,title,character
0,char_1,á
1,char_2,é
2,char_3,í
3,char_4,ó
4,char_5,ú
5,char_6,ü


Si los datos se procesan incorrectamente, pueden aparecer caracteres como "�". También es común lidiar con varios alfabetos o idiomas simultáneamente, por lo que se recomienda unificar el procesamiento en UTF-8 y, si es necesario, utilizar bibliotecas para detectar el idioma, como langdetect, langid, o APIs como Google Translator o los nuevos LLMs como Gemini, OpenAI o Claude.

Un ejemplo para unificar la codificación para el procesamiento de datos:

In [24]:
def convertir_latin1_a_utf8(ruta_entrada, ruta_salida):
    with open(ruta_entrada, "r", encoding="latin-1") as f_in:
        contenido = f_in.read()
    with open(ruta_salida, "w", encoding="utf-8") as f_out:
        f_out.write(contenido)

convertir_latin1_a_utf8("datos_latin1.csv", "datos_utf-8.csv")

## 2.4 Identificación de Codificaciones con la Librería chardet
La librería `chardet` es útil para identificar el tipo de codificación de un texto y se instala con pip. A continuación se muestra un ejemplo didáctico de cómo identificar múltiples tipos de codificación con esta librería:

In [25]:
import chardet
import os

for f in os.listdir():
    if "test" in f:
        print(f)
        with open(f, "br") as RF:
            t = RF.read()
            detection = chardet.detect(t)
            print(detection)
            encoding = detection["encoding"]
            print(t.decode(encoding))

test_utf16.txt
{'encoding': 'UTF-16', 'confidence': 1.0, 'language': ''}
Hola, ¿cómo estás?
test-cyrilic.txt
{'encoding': 'ISO-8859-5', 'confidence': 0.993367969782052, 'language': 'Bulgarian'}
Строка 1: Это пример строки текста.
test_utf8.txt
{'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}
Hola, ¿cómo estás?
This is an additional line.
Voici une autre ligne de texte.
Hier ist eine weitere Textzeile.
这里有另一行文字。
ここに別の行があります。
여기에 또 다른 줄이 있습니다.
Это еще одна строка текста.
Aqui está outra linha de texto.

test_ascii.txt
{'encoding': 'ascii', 'confidence': 1.0, 'language': ''}
This is an additional line.

test_latin-1.txt
{'encoding': 'ISO-8859-1', 'confidence': 0.73, 'language': ''}
Hola, ¿Como estás?


# 3 Introducción a las Expresiones Regulares
Las expresiones regulares (regex) son herramientas poderosas para buscar, filtrar y manipular cadenas de texto basadas en patrones específicos. Son esenciales en la minería de texto para tareas como limpieza, normalización y extracción de información.

## 3.1 Estructura de las Expresiones Regulares
- **Caracteres especiales**:
  - `.`: Coincide con cualquier carácter excepto el salto de línea.
  - `^` y `$`: Delimitan el inicio y fin de la línea.
  - `*`, `+`, `?`: Modificadores de repetición.
  - `[]`: Define clases de caracteres, como `[0-9]` para cifras.
  - `\d`, `\w`, `\s`: Atajos para dígitos, caracteres de palabra y espacios en blanco.
- **Agrupaciones y rangos**:
  - `( )`: Agrupa un patrón y permite capturar resultados.
  - `{n,m}`: Define un rango de repeticiones.

## 3.2 Uso Básico en Python
El módulo `re` de Python proporciona funciones esenciales:
- `re.match(patron, cadena)`: Busca al principio del string.
- `re.search(patron, cadena)`: Busca la primera aparición del patrón.
- `re.findall(patron, cadena)`: Devuelve una lista con todas las apariciones.
- `re.sub(patron, reemplazo, cadena)`: Reemplaza todas las apariciones del patrón.
- `re.split(patron, cadena)`: Divide el string utilizando el patrón como delimitador.

## 3.3 Ejemplos Prácticos
### Encontrar Hashtags en Tweets

In [26]:
import re
import pandas as pd

regex_hashtag = re.compile(r"#\w+")
tweets = ["Amo la ciencia de datos! #DataScience #Python", "Me encanta INSD en la U-Tad! #WorkHard #Focus"]

df_pandas = pd.DataFrame({"text": tweets})
df_pandas["hashtags"] = df_pandas["text"].str.findall(regex_hashtag)
print(df_pandas)

                                            text                 hashtags
0  Amo la ciencia de datos! #DataScience #Python  [#DataScience, #Python]
1  Me encanta INSD en la U-Tad! #WorkHard #Focus      [#WorkHard, #Focus]


### Limpiar Texto

In [27]:
import re
import pandas as pd

def limpiar_texto(texto):
    texto_limpio = re.sub(r"\.{2,}", ".", texto)  # Eliminar signos de puntuación repetidos
    texto_limpio = re.sub(r"\s{2,}", " ", texto_limpio)  # Reemplazar múltiples espacios por uno
    return texto_limpio

tweets = ["Amo    la    ciencia... de datos! #DataScience #Python", "Me    encanta... INSD en      la U-Tad! #WorkHard #Focus"]
df_pandas = pd.DataFrame({"texto original": tweets})
df_pandas["texto limpio"] = df_pandas["texto original"].apply(limpiar_texto)
print(df_pandas)

                                      texto original  \
0  Amo    la    ciencia... de datos! #DataScience...   
1  Me    encanta... INSD en      la U-Tad! #WorkH...   

                                     texto limpio  
0  Amo la ciencia. de datos! #DataScience #Python  
1  Me encanta. INSD en la U-Tad! #WorkHard #Focus  


### Extraer Números de Serie

In [28]:
import re
import pandas as pd

exp_serie = re.compile(r"[A-Z]{3}-\d{4}-[A-Z]{3}")
lineas = ["Producto: ABC-1234-XYZ fecha: 2022-10-10", "ID: ZZZ-9999-AAA   Producto: Ordenador", "Texto irrelevante: AAA 3333 BBB"]

def extract_serial(line):
    match = exp_serie.findall(line)
    return match if match else None

df_pandas = pd.DataFrame({"text": lineas})
df_pandas["serial number"] = df_pandas["text"].apply(extract_serial)
print(df_pandas)

                                       text   serial number
0  Producto: ABC-1234-XYZ fecha: 2022-10-10  [ABC-1234-XYZ]
1    ID: ZZZ-9999-AAA   Producto: Ordenador  [ZZZ-9999-AAA]
2           Texto irrelevante: AAA 3333 BBB            None


# 4: Librerías re y regex en Python
## 4.1 Introducción
En minería de texto, las expresiones regulares son esenciales para la limpieza y extracción de información. Aunque el módulo `re` de Python es útil, la librería `regex` ofrece funcionalidades adicionales y mejoras, especialmente para documentos con caracteres poco usuales.

## 4.2 Limitaciones de re
El módulo `re` tiene algunas limitaciones:
- **Lookbehind fijo**: `re` requiere que la longitud de la subexpresión en lookbehind sea fija.
- **Compatibilidad**: No implementa todos los aspectos de la sintaxis de expresiones regulares de otros lenguajes.
- **Rendimiento**: Puede ser menos eficiente en patrones complejos.

## 4.3 Diferencias entre re y regex
La librería `regex` es un reemplazo mejorado de `re`, ofreciendo:

### 4.3.1 Lookbehind
#### Ejemplo con regex

In [29]:
import regex

pat = regex.compile(r"(?<=\d+)car")
texto = "123car 55car 9999car"
resultados = pat.findall(texto)
print("Coincidencias (regex):", resultados)

Coincidencias (regex): ['car', 'car', 'car']


**Explicación**:
- `(?<=\d+)`: Este es un lookbehind que busca una secuencia de uno o más dígitos (`\d+`) antes de la palabra "car". En `regex`, el lookbehind puede tener longitud variable.
- `car`: La palabra que queremos encontrar después de los dígitos.

**Resultado**: Encuentra todas las ocurrencias de "car" que están precedidas por uno o más dígitos.

### 4.3.2 Soporte Unicode avanzado
#### Ejemplo con regex

In [30]:
import regex

pattern_emoji = regex.compile(r"\p{Extended_Pictographic}+")
texto_emojis = "Hoy me siento feliz 😄 y triste 😢 a la vez."
emojis = pattern_emoji.findall(texto_emojis)
print(emojis)

['😄', '😢']


**Explicación**:
- `\p{Extended_Pictographic}`: Esta es una propiedad Unicode que coincide con caracteres pictográficos extendidos, como emojis.
- `+`: Modificador que indica que queremos encontrar una o más ocurrencias consecutivas de emojis.

**Resultado**: Encuentra todos los emojis en el texto.

### 4.3.3 Recursividad y anidación
#### Ejemplo con regex

In [31]:
import regex

par_patron = regex.compile(r"""
    \(
       (?: [^()]+ | (?R) )*
    \)
""", regex.VERBOSE)

texto_par = "Aquí (tenemos (un ejemplo) de (anidación (compleja))) y (otro)."
encontrados = par_patron.findall(texto_par)
print("Paréntesis anidados:", encontrados)

Paréntesis anidados: ['(tenemos (un ejemplo) de (anidación (compleja)))', '(otro)']


**Explicación**:
- `\(` y `\)`: Coinciden con los caracteres de paréntesis de apertura y cierre.
- `(?: [^()]+ | (?R) )*`: Esta es una expresión regular recursiva:
  - `[^()]+`: Coincide con cualquier secuencia de caracteres que no sean paréntesis.
  - `(?R)`: Llama recursivamente a la expresión regular completa, permitiendo la coincidencia de paréntesis anidados.
- `regex.VERBOSE`: Permite escribir la expresión regular en múltiples líneas con comentarios para mayor claridad.

**Resultado**: Encuentra todas las secuencias de paréntesis anidados en el texto.

## 4.4 Casos de uso con re y regex
### 4.4.1 Obtener valores numéricos con separadores
#### Ejemplo con regex

In [32]:
import regex

pattern_numerico = regex.compile(r"(?<!\w)(\d+(?:[.,]\d+)?)(?!\w)")
texto_mixto = "Precio: 45,67 euros, o tal vez 100.50 USD, etc. Valor2=123"
coinc = pattern_numerico.findall(texto_mixto)
print(coinc)

['45,67', '100.50', '123']


**Explicación**:
- `(?<!\w)`: Lookbehind negativo que asegura que el número no esté precedido por un carácter de palabra.
- `\d+`: Coincide con una secuencia de uno o más dígitos.
- `(?:[.,]\d+)?`: Grupo no capturante que coincide con un punto o una coma seguido de uno o más dígitos. El `?` indica que esta parte es opcional.
- `(?!\w)`: Lookahead negativo que asegura que el número no esté seguido por un carácter de palabra.

**Resultado**: Encuentra todos los números con separadores decimales en el texto.

### 4.4.2 Extracción de subdominios
#### Ejemplo con regex

In [33]:
import regex

dom_pat = regex.compile(r"(?<=https?://)([\w.-]+)\.(\p{Letter}{2,})(?=/?)")
test_urls = [
    "https://sub.example.com/path/to/file",
    "http://mydomain.org/index.html",
    "https://xxx.co/short"
]

for url in test_urls:
    res = dom_pat.search(url)
    if res:
        print(f"URL: {url}")
        print(" Dominio principal:", res.group(1))
        print(" TLD:", res.group(2))

URL: https://sub.example.com/path/to/file
 Dominio principal: sub.example
 TLD: com
URL: http://mydomain.org/index.html
 Dominio principal: mydomain
 TLD: org
URL: https://xxx.co/short
 Dominio principal: xxx
 TLD: co


**Explicación**:
- `(?<=https?://)`: Lookbehind que asegura que la coincidencia esté precedida por "http://" o "https://".
- `([\w.-]+)`: Grupo que coincide con una secuencia de caracteres de palabra, puntos o guiones.
- `\.`: Coincide literalmente con un punto.
- `\p{Letter}{2,}`: Propiedad Unicode que coincide con dos o más letras.
- `(?=/?)`: Lookahead que asegura que la coincidencia esté seguida opcionalmente por una barra.

**Resultado**: Extrae el dominio principal y la TLD de las URLs.

### 4.4.3 Detección de texto específico
#### Ejemplo con regex

In [34]:
import regex

pattern_cyr = regex.compile(r"\p{Script=Cyrillic}+")
sample_text = "Texto con кириллица y latín mezclado"
found_cyr = pattern_cyr.findall(sample_text)
print("Cirílico:", found_cyr)

Cirílico: ['кириллица']


**Explicación**:
- `\p{Script=Cyrillic}`: Propiedad Unicode que coincide con caracteres del alfabeto cirílico.
- `+`: Modificador que indica que queremos encontrar una o más ocurrencias consecutivas de caracteres cirílicos.

**Resultado**: Encuentra todas las secuencias de caracteres cirílicos en el texto.

## 4.5 Cuándo usar re o regex
La elección entre `re` y `regex` depende de:
- **Patrones complejos**: `regex` es mejor para anidamiento o lookbehind variable.
- **Rendimiento**: `regex` puede ser más eficiente en ciertos casos.
- **Unicode avanzado**: `regex` maneja mejor las propiedades Unicode.

Ambas librerías se integran bien con pandas y polars para la manipulación tabular del texto, y son útiles antes de pasar a librerías de procesamiento de lenguaje natural como NLTK o spaCy.